# pruning-resnet18 — Thí nghiệm của phần pruning/ trên Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NhatMinh1503/pruning-resnet18/blob/main/pruning/colab_pruning.ipynb)

Notebook này chạy các thí nghiệm của `pruning/` (Hoa). Cần có `resnet18_baseline.pt` trên Drive chung.

| Bước | Nội dung | Thời gian (T4, ước lượng) |
|---|---|---|
| 3 | `ablate_layers.py`: giữ dense fc / conv1 / downsample (Q2) | ~2 phút |
| 4 | `ablate_stages.py`: chỉ cắt layer1..4, cấu hình matched (Q2) | ~2 phút |
| 5 | `run_structured.py`: structured 25/50/75% + fine-tune 10 epoch (Q5) | ~15 phút |
| 6 | `plot_tradeoff.py`: vẽ hình | vài giây |

**Chuẩn bị 1 lần:** giống `training/colab_training.ipynb` (thêm lối tắt `pruning-project` vào *Drive của tôi*, chọn GPU T4).

## 1. Kết nối Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT = '/content/drive/MyDrive/pruning-project/checkpoints'
RESULTS = '/content/drive/MyDrive/pruning-project/results'
BASE = f'{CKPT}/resnet18_baseline.pt'
!ls -lh {BASE}

## 2. Lấy code mới nhất + kiểm tra GPU

In [ ]:
import os
BRANCH = 'main'   # nếu PR pruning chưa merge thì đổi thành 'pruning'
if os.path.isdir('/content/pruning-resnet18'):
    %cd /content/pruning-resnet18
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/NhatMinh1503/pruning-resnet18.git /content/pruning-resnet18
    %cd /content/pruning-resnet18
!pip install -q thop
!nvidia-smi --query-gpu=name --format=csv
!python pruning/prune.py && python pruning/structured.py

## 3. Ablation theo lớp nhỏ (Q2)

In [ ]:
!python pruning/ablate_layers.py --checkpoint {BASE} --data-dir /content/data

## 4. Ablation theo stage + matched sparsity (Q2)

In [ ]:
!python pruning/ablate_stages.py --checkpoint {BASE} --data-dir /content/data

## 5. Structured pruning + fine-tune (Q5)

Checkpoint lưu vào Drive với tên `resnet18_structured_{ratio}_finetuned.pt` (đọc lại bằng `pruning.structured.load_structured`).

In [ ]:
!python pruning/run_structured.py --checkpoint {BASE} --data-dir /content/data --ckpt-dir {CKPT}

## 6. Vẽ hình

In [ ]:
!python pruning/plot_tradeoff.py
from IPython.display import Image, display
display(Image('results/figures/q2_q5_tradeoff.png'))

## 7. Lưu kết quả lên Drive chung (rồi commit các CSV vào Git từ máy mình)

In [ ]:
!mkdir -p {RESULTS}
!cp results/ablation_layers.csv results/ablation_stages.csv results/structured.csv {RESULTS}/
!ls -lh {RESULTS}